In [13]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from baseline import *
from cvmodeling import *
from utils import *
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
import seaborn as sns
sns.set_style("whitegrid")
from sklearn.model_selection import GridSearchCV, cross_val_score, KFold, StratifiedKFold, RandomizedSearchCV
from sklearn.metrics import roc_auc_score, roc_curve, precision_recall_curve, precision_score, recall_score, f1_score, accuracy_score, confusion_matrix, classification_report, make_scorer, auc
from sklearn.model_selection import KFold, StratifiedKFold, GridSearchCV, cross_val_score, cross_validate, train_test_split

In [14]:
data = pd.read_csv('House_Rent_Dataset.csv')

In [15]:
data

,Posted On,BHK,Rent,Size,Floor,Area Type,Area Locality,City,Furnishing Status,Tenant Preferred,Bathroom,Point of Contact
0,2022-05-18,2,10000,1100,Ground out of 2,Super Area,Bandel,Kolkata,Unfurnished,Bachelors/Family,2,Contact Owner
1,2022-05-13,2,20000,800,1 out of 3,Super Area,"Phool Bagan, Kankurgachi",Kolkata,Semi-Furnished,Bachelors/Family,1,Contact Owner
2,2022-05-16,2,17000,1000,1 out of 3,Super Area,Salt Lake City Sector 2,Kolkata,Semi-Furnished,Bachelors/Family,1,Contact Owner
3,2022-07-04,2,10000,800,1 out of 2,Super Area,Dumdum Park,Kolkata,Unfurnished,Bachelors/Family,1,Contact Owner
4,2022-05-09,2,7500,850,1 out of 2,Carpet Area,South Dum Dum,Kolkata,Unfurnished,Bachelors,1,Contact Owner
...,...,...,...,...,...,...,...,...,...,...,...,...
4741,2022-05-18,2,15000,1000,3 out of 5,Carpet Area,Bandam Kommu,Hyderabad,Semi-Furnished,Bachelors/Family,2,Contact Owner
4742,2022-05-15,3,29000,2000,1 out of 4,Super Area,"Manikonda, Hyderabad",Hyderabad,Semi-Furnished,Bachelors/Family,3,Contact Owner
4743,2022-07-10,3,35000,1750,3 out of 5,Carpet Area,"Himayath Nagar, NH 7",Hyderabad,Semi-Furnished,Bachelors/Family,3,Contact Agent
4744,2022-07-06,3,45000,1500,23 out of 34,Carpet Area,Gachibowli,Hyderabad,Semi-Furnished,Family,2,Contact Agent


In [16]:
data.isnull().sum()

Posted On            0
BHK                  0
Rent                 0
Size                 0
Floor                0
Area Type            0
Area Locality        0
City                 0
Furnishing Status    0
Tenant Preferred     0
Bathroom             0
Point of Contact     0
dtype: int64

In [17]:
# Regular expression to extract the current floor and total number of floors
pattern = r'(?P<CurrentFloor>\w+)\s*out\s*of\s*(?P<TotalFloors>\d+)'

# Extract the values into new columns
data[['CurrentFloor', 'TotalFloors']] = data['Floor'].str.extract(pattern)

# Replace non-numeric floor labels with specific values
floor_replacements = {
    'Ground': 0,
    'Basement': -1  # Assuming 'Basement' should be treated as floor -1
}

data['CurrentFloor'] = data['CurrentFloor'].replace(floor_replacements)

# Convert columns to appropriate data types
data['CurrentFloor'] = pd.to_numeric(data['CurrentFloor'], errors='coerce')
data['TotalFloors'] = pd.to_numeric(data['TotalFloors'], errors='coerce')
data = data.drop(columns=['Floor'])

In [18]:
data['Year'] = pd.to_datetime(data['Posted On']).dt.year
data = data.drop(columns=['Posted On'])
data.head()

,BHK,Rent,Size,Area Type,Area Locality,City,Furnishing Status,Tenant Preferred,Bathroom,Point of Contact,CurrentFloor,TotalFloors,Year
0,2,10000,1100,Super Area,Bandel,Kolkata,Unfurnished,Bachelors/Family,2,Contact Owner,0.0,2.0,2022
1,2,20000,800,Super Area,"Phool Bagan, Kankurgachi",Kolkata,Semi-Furnished,Bachelors/Family,1,Contact Owner,1.0,3.0,2022
2,2,17000,1000,Super Area,Salt Lake City Sector 2,Kolkata,Semi-Furnished,Bachelors/Family,1,Contact Owner,1.0,3.0,2022
3,2,10000,800,Super Area,Dumdum Park,Kolkata,Unfurnished,Bachelors/Family,1,Contact Owner,1.0,2.0,2022
4,2,7500,850,Carpet Area,South Dum Dum,Kolkata,Unfurnished,Bachelors,1,Contact Owner,1.0,2.0,2022


In [19]:
from sklearn.preprocessing import LabelEncoder
label_encoder = LabelEncoder()
data['Area Type'] = label_encoder.fit_transform(data['Area Type'])
data['Area Locality'] = label_encoder.fit_transform(data['Area Locality'])
data['City'] = label_encoder.fit_transform(data['City'])
data['Furnishing Status'] = label_encoder.fit_transform(data['Furnishing Status'])
data['Tenant Preferred'] = label_encoder.fit_transform(data['Tenant Preferred'])
data['Point of Contact'] = label_encoder.fit_transform(data['Point of Contact'])

data.head()

,BHK,Rent,Size,Area Type,Area Locality,City,Furnishing Status,Tenant Preferred,Bathroom,Point of Contact,CurrentFloor,TotalFloors,Year
0,2,10000,1100,2,221,4,2,1,2,2,0.0,2.0,2022
1,2,20000,800,2,1527,4,1,1,1,2,1.0,3.0,2022
2,2,17000,1000,2,1760,4,1,1,1,2,1.0,3.0,2022
3,2,10000,800,2,526,4,2,1,1,2,1.0,2.0,2022
4,2,7500,850,1,1890,4,2,0,1,2,1.0,2.0,2022


In [20]:
# from sklearn.model_selection import train_test_split

# y = data['Rent']
# X = data.drop(columns=['Rent'])
# # X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, shuffle=True, test_size=0.3)

In [22]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
scaler.fit(data.drop(columns=['Rent']))

data_scaled = scaler.transform(data.drop(columns=['Rent']))

In [25]:
data_scaled = pd.DataFrame(data_scaled, columns=data.drop(columns=['Rent']).columns)

In [26]:
data_scaled['Rent'] = data['Rent']

In [27]:
data_scaled.head()

,BHK,Size,Area Type,Area Locality,City,Furnishing Status,Tenant Preferred,Bathroom,Point of Contact,CurrentFloor,TotalFloors,Year,Rent
0,-0.100773,0.208960,0.968881,-1.375251,0.863449,1.111575,0.145534,0.038594,0.68966,-0.595896,-0.525248,0.0,10000
1,-0.100773,-0.264125,0.968881,0.687519,0.863449,-0.349387,0.145534,-1.092067,0.68966,-0.422687,-0.419637,0.0,20000
2,-0.100773,0.051265,0.968881,1.055532,0.863449,-0.349387,0.145534,-1.092067,0.68966,-0.422687,-0.419637,0.0,17000
3,-0.100773,-0.264125,0.968881,-0.893517,0.863449,1.111575,0.145534,-1.092067,0.68966,-0.422687,-0.525248,0.0,10000
4,-0.100773,-0.185277,-1.028647,1.260862,0.863449,1.111575,-1.783809,-1.092067,0.68966,-0.422687,-0.525248,0.0,7500


In [28]:
data.to_csv("reg_data.csv", index=False)